In [25]:
# Data Science Project: Individual Planning Report
# Student: Ryan Wong
# Date: November 14

# Load required libraries
library(tidyverse)
library(knitr)

# Load the data
players <- read_csv("dsci100-project-004-10/players.csv")
sessions <- read_csv("dsci100-project-004-10/sessions.csv")

# Display basic information
cat("Players dataset dimensions:", dim(players)[1], "rows,", dim(players)[2], "columns\n")
cat("Sessions dataset dimensions:", dim(sessions)[1], "rows,", dim(sessions)[2], "columns\n")

Rows: 196 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): experience, hashedEmail, name, gender
dbl (2): played_hours, Age
lgl (1): subscribe

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1535 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): hashedEmail, start_time, end_time
dbl (2): original_start_time, original_end_time

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Players dataset dimensions: 196 rows, 7 columns
Sessions dataset dimensions: 1535 rows, 5 columns


# 1. Data Description
The data wass collected from a UBC Minecraft research server. Each player's activity is automatically logged, including session start/end times, duration, and interactions. Player demographic information (age, gender, experience level) was self-reported.
### Data Sources
The dataset consists of two CSV files:
- **players.csv**: Player demographic and profile information
- **sessions.csv**: Individual gameplay session records

In [26]:
# Summary table for players dataset
player_vars <- data.frame(
  Variable = names(players),
  Type = sapply(players, function(x) class(x)[1]),
  Description = c(
    "Player experience level (Beginner, Amateur, Regular, Veteran, Pro)",
    "Whether player subscribed to newsletter (TRUE/FALSE)",
    "Anonymized email identifier",
    "Total hours played on the server",
    "Player name",
    "Player gender identity",
    "Player age in years"
  ),
  Missing = sapply(players, function(x) sum(is.na(x)))
)
kable(player_vars, caption = "Players Dataset Variable Summary")



Table: Players Dataset Variable Summary

|             |Variable     |Type      |Description                                                        | Missing|
|:------------|:------------|:---------|:------------------------------------------------------------------|-------:|
|experience   |experience   |character |Player experience level (Beginner, Amateur, Regular, Veteran, Pro) |       0|
|subscribe    |subscribe    |logical   |Whether player subscribed to newsletter (TRUE/FALSE)               |       0|
|hashedEmail  |hashedEmail  |character |Anonymized email identifier                                        |       0|
|played_hours |played_hours |numeric   |Total hours played on the server                                   |       0|
|name         |name         |character |Player name                                                        |       0|
|gender       |gender       |character |Player gender identity                                             |       0|
|Age         

In [27]:
# Summary table for sessions dataset
session_vars <- data.frame(
  Variable = names(sessions),
  Type = sapply(sessions, function(x) class(x)[1]),
  Description = c(
    "Anonymized email identifier (links to players table)",
    "Session start time (formatted)",
    "Session end time (formatted)",
    "Session start time (Unix timestamp)",
    "Session end time (Unix timestamp)"
  ),
  Missing = sapply(sessions, function(x) sum(is.na(x)))
)

kable(session_vars, caption = "Sessions Dataset Variable Summary")



Table: Sessions Dataset Variable Summary

|                    |Variable            |Type      |Description                                          | Missing|
|:-------------------|:-------------------|:---------|:----------------------------------------------------|-------:|
|hashedEmail         |hashedEmail         |character |Anonymized email identifier (links to players table) |       0|
|start_time          |start_time          |character |Session start time (formatted)                       |       0|
|end_time            |end_time            |character |Session end time (formatted)                         |       2|
|original_start_time |original_start_time |numeric   |Session start time (Unix timestamp)                  |       0|
|original_end_time   |original_end_time   |numeric   |Session end time (Unix timestamp)                    |       2|

In [28]:
# Summary statistics
cat("\n**Dataset Sizes:**\n")
cat("- Total unique players:", nrow(players), "\n")
cat("- Total gameplay sessions:", nrow(sessions), "\n")
cat("- Sessions with missing end times:", sum(is.na(sessions$end_time)), "\n")

# Distribution of categorical variables
cat("\n**Experience Level Distribution:**\n")
print(table(players$experience))

cat("\n**Gender Distribution:**\n")
print(table(players$gender))

cat("\n**Subscription Rate:**\n")
print(table(players$subscribe))


**Dataset Sizes:**
- Total unique players: 196 
- Total gameplay sessions: 1535 
- Sessions with missing end times: 2 

**Experience Level Distribution:**

 Amateur Beginner      Pro  Regular  Veteran 
      63       35       14       36       48 

**Gender Distribution:**

          Agender            Female              Male        Non-binary 
                2                37               124                15 
            Other Prefer not to say      Two-Spirited 
                1                11                 6 

**Subscription Rate:**

FALSE  TRUE 
   52   144 


In [24]:
# Calculate means for quantitative variables
quantitative_means <- data.frame(
  Variable = c("played_hours", "Age"),
  Mean = c(
    round(mean(players$played_hours, na.rm = TRUE), 2),
    round(mean(as.numeric(players$Age), na.rm = TRUE), 2)
  ),
  SD = c(
    round(sd(players$played_hours, na.rm = TRUE), 2),
    round(sd(as.numeric(players$Age), na.rm = TRUE), 2)
  ),
  Min = c(
    round(min(players$played_hours, na.rm = TRUE), 2),
    round(min(as.numeric(players$Age), na.rm = TRUE), 2)
  ),
  Max = c(
    round(max(players$played_hours, na.rm = TRUE), 2),
    round(max(as.numeric(players$Age), na.rm = TRUE), 2)
  ),
  N_Missing = c(
    sum(is.na(players$played_hours)),
    sum(is.na(as.numeric(players$Age)))
  )
)

kable(quantitative_means, caption = "Summary Statistics for Quantitative Variables")



Table: Summary Statistics for Quantitative Variables

|Variable     |  Mean|    SD| Min|   Max| N_Missing|
|:------------|-----:|-----:|---:|-----:|---------:|
|played_hours |  5.85| 28.36|   0| 223.1|         0|
|Age          | 21.14|  7.39|   9|  58.0|         2|

### Data Quality Issues

**Observed Issues:**
- 2 sessions have missing end times (incomplete sessions)
- Age variable contains "NA" strings that need conversion
- Played_hours shows extreme outliers (maximum > 200 hours)
- Gender has multiple categories with small sample sizes

**Potential Issues:**
- Self-reported demographic data may have accuracy concerns
- Session timestamps need timezone considerations
- Some players have zero sessions recorded